In [5]:
import pandas as pd

In [6]:
# Đọc dữ liệu từ file CSV ban đầu và tạo một bản sao để tiến hành làm sạch
df_shipments_raw = pd.read_csv('shipments_realistic.csv')
df_shipments_silver = df_shipments_raw.copy()

In [7]:
# Kiểm tra và loại bỏ các dòng dữ liệu bị trùng lặp
df_shipments_silver = df_shipments_silver.drop_duplicates()

In [8]:
# Lấy danh sách các cột chứa chuỗi (text/object)
str_cols = df_shipments_silver.select_dtypes(include=['object']).columns

In [9]:
# Lặp qua từng cột chuỗi để xóa khoảng trắng thừa ở hai đầu
for col in str_cols:
    df_shipments_silver[col] = df_shipments_silver[col].astype(str).str.strip()

In [10]:
# Khai báo các cột cần chuyển đổi sang định dạng thời gian
date_cols = ['ship_date', 'delivery_date']
# Chuyển đổi sang datetime, các giá trị không hợp lệ sẽ bị ép thành NaT (errors='coerce')
for col in date_cols:
    if col in df_shipments_silver.columns:
        df_shipments_silver[col] = pd.to_datetime(df_shipments_silver[col], errors='coerce')

In [11]:
# Kiểm tra tính hợp lý của dữ liệu: Ngày giao hàng phải diễn ra sau hoặc cùng ngày gửi
if 'ship_date' in df_shipments_silver.columns and 'delivery_date' in df_shipments_silver.columns:
    valid_dates = df_shipments_silver['delivery_date'] >= df_shipments_silver['ship_date']
    # Giữ lại các dòng có ngày hợp lệ hoặc các đơn hàng chưa được giao (delivery_date bị trống/isna)
    df_shipments_silver = df_shipments_silver[valid_dates | df_shipments_silver['delivery_date'].isna()]

In [12]:
# Xuất dữ liệu đã được làm sạch ra file CSV mới
df_shipments_silver.to_csv('silver_shipments_realistic.csv', index=False)

# Hiển thị vài dòng đầu tiên của dữ liệu để kiểm tra kết quả
display(df_shipments_silver.head())

,shipper_id,order_id,ship_date,delivery_date,shipping_fee,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,...,join_date,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,region,district
0,SHP00001,1,2012-07-07,2012-07-11,1.37,Viettel Post,Truck,7,5.0,99.0,...,2026-03-17,Bùi Văn Long,991476209,Male,27,Married,Bachelor,Phan Rang-Thap Cham,Central,District #25
1,SHP00002,2,2012-07-06,2012-07-10,2.60,J&T Express,Van,2,4.9,98.4,...,2025-01-29,Trần Anh Khánh,959297982,Male,41,Married,Bachelor,Phan Thiet,Central,District #29
2,SHP00003,3,2012-07-04,2012-07-07,2.38,GHN,Motorbike,10,4.8,95.1,...,2019-11-13,Hoàng Thị Khánh,927142576,Male,30,Single,High School,Long Xuyen,West,District #34
3,SHP00004,4,2012-07-05,2012-07-11,2.49,Viettel Post,Truck,8,5.0,96.3,...,2025-12-22,Trần Đức Vy,971617475,Female,42,Married,College,Kon Tum,Central,District #27
4,SHP00005,6,2012-07-09,2012-07-16,25.79,BEST Express,Truck,10,4.6,95.7,...,2019-12-19,Trần Minh Cường,979196342,Male,31,Married,College,Da Nang,Central,District #23
